# ជំពូក ៥៖ Pretraining លើទិន្នន័យគ្មានស្លាក

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/ch05.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**កូដ៖** តក្កវិទ្យាកូដ (logic) ដូច upstream ទាំងស្រុង — មានតែ *comments* ក្នុងកូដដែលត្រូវបានបកប្រែជាភាសាខ្មែរដោយសហគមន៍; កូដដែល comment ជាប់ (commented-out code) នៅដដែល។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ០. ការត្រៀមបរិស្ថាន
- ## ១. ការវាយតម្លៃម៉ូដែលដើម
- ## ២. Cross-entropy loss
- ## ៣. ការផ្ទុកទិន្នន័យ និងការបំបែក train/val
- ## ៤. អនុគមន៍បណ្តុះបណ្តាល
- ## ៥. ការគូសក្រាផ loss
- ## ៦. Sampling និង temperature
- ## ៧. Top-k និងអនុគមន៍ generate
- ## ៨. ការរក្សាទុក និងផ្ទុក weights
- ## ៩. ការផ្ទុក weights GPT-2 pretrained
- ## ១០. ការផ្គូផ្គាប់ weights ចូលម៉ូដែលរបស់យើង

In [ ]:
# ការតភ្ជាប់ឯកសារជំនួយ — ឱ្យដំណើរការភ្ជាប់ដូច notebook ដើមរបស់ upstream
# (ចង្អុល sys.path និងថតធ្វើការទៅកាន់ថតកូដក្នុងឃ្លាំងកសារខ្មែរនេះ)
import os, sys
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "កូដ")):
    _p = os.path.dirname(_p)
_UP = os.path.join(_p, "កូដ", "ch05/01_main-chapter-code")
sys.path.insert(0, _UP)
os.chdir(_UP)


## ០. ការត្រៀមបរិស្ថាន

ពិនិត្យកំណែ torch/tiktoken និងនាំចូលម៉ូឌុលពីជំពូកមុន។

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib", 
        "numpy", 
        "tiktoken", 
        "torch",
        "tensorflow"  # សម្រាប់ weights pretrained របស់ OpenAI
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
import torch
from previous_chapters import GPTModel
# បើឯកសារ `previous_chapters.py` មិនមានក្នុងមូលដ្ឋានទេ អ្នកអាច import វាពី package PyPI `llms-from-scratch`។
# សម្រាប់លម្អិត សូមមើល៖ https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# ឧទាហរណ៍៖
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,  # ទំហំ vocabulary
    "context_length": 256,  # ប្រវែង context ខ្លី (ដើម 1024)
    "emb_dim": 768,  # វិមាត្រ embedding
    "n_heads": 12,  # ចំនួន attention heads
    "n_layers": 12,  # ចំនួនស្រទាប់
    "drop_rate": 0.1,  # អត្រា dropout
    "qkv_bias": False  # bias សម្រាប់ query-key-value
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # បិទ dropout ពេលធ្វើ inference

In [ ]:
import tiktoken
from previous_chapters import generate_text_simple

# ជាជម្រើស៖
# from llms_from_scratch.ch04 import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # បន្ថែមវិមាត្រ batch
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # យកវិមាត្រ batch ចេញ
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

## ១. ការវាយតម្លៃម៉ូដែលដើម

គណនា probability នៃ token បន្ទាប់ពីម៉ូដែលដែលមិនទាន់បាន train។

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [ ]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1)  # ប្រូបាបរបស់ token នីមួយៗក្នុង vocabulary
print(probas.shape)  # រូបរាង៖ (batch_size, num_tokens, vocab_size)

In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

In [ ]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

## ២. Cross-entropy loss

គណនា loss ដោយដៃពី log-probabilities រួចប្រៀបធៀបជាមួយ nn.functional.cross_entropy និង perplexity។

In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

In [ ]:
# គណនា logarithm នៃប្រូបាប tokens ទាំងអស់
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

In [ ]:
# គណនាប្រូបាបមធ្យមសម្រាប់ token នីមួយៗ
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

In [ ]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

In [ ]:
# Logits មានរូបរាង (batch_size, num_tokens, vocab_size)
print("Logits shape:", logits.shape)

# Targets មានរូបរាង (batch_size, num_tokens)
print("Targets shape:", targets.shape)

In [ ]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

In [ ]:
perplexity = torch.exp(loss)
print(perplexity)

## ៣. ការផ្ទុកទិន្នន័យ និងការបំបែក train/val

ផ្ទុកអត្ថបទ បំបែកជាផ្នែក train/val និងបង្កើត dataloaders។

In [ ]:
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


# សៀវភៅដើមបានប្រើកូដខាងក្រោមនេះ
# ប៉ុន្តែ urllib ប្រើការកំណត់ protocol ចាស់ ដែលអាចបង្កបញ្ហាដល់អ្នកប្រើមួយចំនួនដែលប្រើ VPN។
# កំណែ `requests` ខាងលើមានភាពរឹងមាំជាង urllib ដែលជាទូទៅដំណើរការល្អ ប៉ុន្តែអ្នកអានខ្លះបានរាយការណ៍ពីបញ្ហា urllib ពេលប្រើ VPN (របស់ក្រុមហ៊ុន)។

        
# import os
# import urllib.request

# file_path = "the-verdict.txt"
# url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

# if not os.path.exists(file_path):
#     with urllib.request.urlopen(url) as response:
#         text_data = response.read().decode('utf-8')
#     with open(file_path, "w", encoding="utf-8") as file:
#         file.write(text_data)
# else:
#     with open(file_path, "r", encoding="utf-8") as file:
#         text_data = file.read()

In [ ]:
# តួអក្សរ ៩៩ ដំបូង
print(text_data[:99])

In [ ]:
# តួអក្សរ ៩៩ ចុងក្រោយ
print(text_data[-99:])

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

In [ ]:
from previous_chapters import create_dataloader_v1
# ជាជម្រើស៖
# from llms_from_scratch.ch02 import create_dataloader_v1

# សមាមាត់ train/validation
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# ការពិនិត្យបឋម

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

## ៤. អនុគមន៍បណ្តុះបណ្តាល

សរសេរ calc_loss_batch, calc_loss_loader និង train_model_simple។

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # បន្ថយចំនួន batches ឱ្យស្ើនឹងចំនួន batches សរុបក្នុង data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # ប្រើ PyTorch 2.9 ឬថ្មីជាងនេះ សម្រាប់លទ្ធផល mps ស្ថិរ
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")


print(f"Using {device} device.")


model.to(device)


torch.manual_seed(123)  # សម្រាប់ភាពស្របគ្នា ដោយសារការច្របល់ក្នុង data loader

with torch.no_grad():  # បិទការតាមដាន gradient សម្រាប់ប្រសិទ្ធភាព ព្រោះយើងមិនកំពុងបណ្តុះបណ្តាលទេ
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # ចាប់ផ្តើមបញ្ជីដើម្បីតាមដាន losses និងចំនួន tokens ដែលបានឃើញ
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # រង្វិលបណ្តុះបណ្តាលមេ
    for epoch in range(num_epochs):
        model.train()  # ដាក់ម៉ូដែលក្នុងរបៀបបណ្តុះបណ្តាល
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()  # កំណត់ gradients នៃ loss ពី iteration មុនឡើងវិញ
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()  # គណនា gradients នៃ loss
            optimizer.step()  # ធ្វើបច្ចុបបន្នភាព weights របស់ម៉ូដែលដោយប្រើ gradients នៃ loss
            tokens_seen += input_batch.numel()
            global_step += 1

            # ជំហានវាយតម្លៃជាជម្រើស
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # បង្ហាញអត្ថបទគំរូបន្ទាប់ពី epoch នីមួយៗ
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # ទ្រង់ទ្រាយបង្ហាញសង្ខេប
    model.train()

In [ ]:
# Note:
# ដោះ comment កូដខាងក្រោមដើម្បីគណនារយៈពេលដំណើរការ
# import time
# start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# ដោះ comment កូដខាងក្រោមដើម្បីបង្ហាញរយៈពេលដំណើរការ
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

## ៥. ការគូសក្រាផ loss

បង្ហាញខ្សែកោង train/val loss តាម epochs។

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # គូសក្រាផ loss នៃ train និង validation ធ្នឹងនឹង epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # បង្ហាញតែស្លាកជាចំនួនគត់លើអ័ក្ស x

    # បង្កើតអ័ក្ស x ទីពីរសម្រាប់ចំនួន tokens ដែលបានឃើញ
    ax2 = ax1.twiny()  # បង្កើតអ័ក្ x ទីពីរដែលចែករំលែកអ័ក្ស y តែមួយ
    ax2.plot(tokens_seen, train_losses, alpha=0)  # ក្រាផមើលមិនឃើញសម្រាប់តម្រឹម ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # លៃតម្រូវ layout ដើម្បីទុកកន្លែង
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

## ៦. Sampling និង temperature

យល់ពីការស្ទូចយក token តាម probability និងឥទ្ធិពលនៃ temperature។

In [ ]:
# ថ្មី ប្រើ CPU នៅទីនេះ ព្រោះ inference ថោកជាមួយម៉ូដែលនេះ និងដើម្បីធានាថាអ្នកអានទទួលបានលទ្ធផលដូចគ្នាក្នុងផ្នែកដែលនៅសល់នៃសៀវភៅ
inference_device = torch.device("cpu")

model.to(inference_device)
model.eval()

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 

inverse_vocab = {v: k for k, v in vocab.items()}

# Suppose input is "every effort moves you", and the LLM
# returns the following logits for the next token:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.argmax(probas).item()

# token បន្ទាប់ដែលបានបង្កើត គឺដូចខាងក្រោម៖
print(inverse_vocab[next_token_id])

In [ ]:
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123)  # កំណត់ seed ដើម្បីភាពស្របគ្នារវាងការរត់
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample), minlength=len(probas))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")

print_sampled_tokens(probas)

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

# តម្លៃ temperature
temperatures = [1, 0.1, 5]  # Original, higher confidence, and lower confidence

# គណនាប្រូបាបដែលបានធ្វើមាត្រដ្ឋាន
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

In [ ]:
# ការគូសក្រាផ
x = torch.arange(len(vocab))
bar_width = 0.15

fig, ax = plt.subplots(figsize=(5, 3))
for i, T in enumerate(temperatures):
    rects = ax.bar(x + i * bar_width, scaled_probas[i], bar_width, label=f'Temperature = {T}')

ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()

plt.tight_layout()
plt.savefig("temperature-plot.pdf")
plt.show()

In [ ]:
print_sampled_tokens(scaled_probas[1])

In [ ]:
print_sampled_tokens(scaled_probas[2])

## ៧. Top-k និងអនុគមន៍ generate

បន្ថែម top-k sampling ចូលក្នុងអនុគមន៍ generate ពេញលេញ។

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)

print("Top logits:", top_logits)
print("Top positions:", top_pos)

In [ ]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float("-inf")), 
    other=next_token_logits
)

print(new_logits)

In [ ]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop ដូចពីមុន៖ យក logits ហើយផ្តោតតែលើ time step ចុងក្រោយ
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # ថ្មី៖ ត្រង logits ដោយ top_k sampling
        if top_k is not None:
            # រក្សាតែតម្លៃ top_k
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # ថ្មី៖ អនុវត្ត temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # ថ្មី (មិនមានក្នុងសៀវភៅ)៖ គន្លឹះស្ថិរភាពលេខ ដើម្បីបានលទ្ធផលស្មើគ្នាលើឧបករណ៍ mps
            # ដកតម្លៃអតិបរមាតាមជួរមុន softmax
            logits = logits - logits.max(dim=-1, keepdim=True).values
            
            # អនុវត្ត softmax ដើម្បីទទួលបានប្រូបាប
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # ស្ទូចយកពីបំណែងចែក
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # ផ្សេងពីនេះដូចពីមុន៖ យក idx នៃធាតុ vocab ដែលមាន logits ខ្ពស់បំផុត
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # បញ្ឈប់ការបង្កើតមុនកំណត់ បើជួប token ចុងបន្សំ ហើយ eos_id ត្រូវបានកំណត់
            break

        # ដូចពីមុន៖ បន្ថែម idx ដែលបានស្ទូចយកទៅបន្សំកំពុងដំណើរការ
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

## ៨. ការរក្សាទុក និងផ្ទុក weights

រក្សាទុក state_dict និង checkpoint (ម៉ូដែល + optimizer) រួចផ្ទុកវិញ។

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
model = GPTModel(GPT_CONFIG_124M)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # ប្រើ PyTorch 2.9 ឬថ្មីជាងនេះ សម្រាប់លទ្ធផល mps ស្ថិរ
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval();

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    }, 
    "model_and_optimizer.pth"
)

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

## ៩. ការផ្ទុក weights GPT-2 pretrained

ទាញយក និងផ្ទុក weights ផ្លូវការរបស់ GPT-2 (124M) ពី OpenAI។

In [ ]:
# pip install tensorflow tqdm

In [ ]:
print("TensorFlow version:", version("tensorflow"))
print("tqdm version:", version("tqdm"))

In [ ]:
# import ធៀបពី gpt_download.py ក្នុងថតនេះ

from gpt_download import download_and_load_gpt2
# ជាជម្រើស៖
# from llms_from_scratch.ch05 import download_and_load_gpt2

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
print("Settings:", settings)

In [ ]:
print("Parameter dictionary keys:", params.keys())

In [ ]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

## ១០. ការផ្គូផ្គាប់ weights ចូលម៉ូដែលរបស់យើង

បំលែង និងដាក់ weights GPT-2 ចូលក្នុង GPTModel ដែលយើងបានសាងសង់។

In [ ]:
# កំណត់រចនាសម្ពន្ធម៉ូដែលក្នុង dictionary ដើម្បីសង្ខេប
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# ចម្លងការកំណត់មូលដ្ឋាន ហើយធ្វើបច្ចុប្បន្នភាពតាមការកំណត់ម៉ូដែលជាក់លាក់
model_name = "gpt2-small (124M)"  # ឈ្មោះម៉ូដែលឧទាហរណ៍
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, 
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, 
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, 
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, 
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, 
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, 
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, 
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, 
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, 
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, 
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])
    
    
load_weights_into_gpt(gpt, params)
gpt.to(device);

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ